# Six Sigma DMAIC — Measure Phase

Baseline data collection and process control chart for IT helpdesk
ticket resolution time (MTTR).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline
sns.set_style("whitegrid")

PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / "data" / "raw"
CHARTS = PROJECT_ROOT / "docs" / "screenshots"
CHARTS.mkdir(parents=True, exist_ok=True)

SLA_HOURS = 24

In [ ]:
df = pd.read_csv(RAW / "helpdesk_tickets.csv", parse_dates=["created_at"])
df = df.sort_values("created_at").reset_index(drop=True)

print(f"{df.shape[0]:,} tickets | {df['category'].nunique()} categories")
df.head()

In [ ]:
print(df.groupby("category")["resolution_hours"].agg(["count", "mean"]).round(1)
        .sort_values("mean", ascending=False))

**Network** has both the highest ticket volume and the highest average resolution time — the clear target for this DMAIC cycle, confirmed with a Pareto chart in the Analyze phase.

### Baseline control chart (Individuals / Moving Range)

In [ ]:
def imr_limits(x):
    """Individuals control chart limits via moving range -- the standard
    SPC method for one-at-a-time continuous data (as opposed to subgrouped
    data, which would use an X-bar/R chart instead)."""
    mr = np.abs(np.diff(x))
    sigma = mr.mean() / 1.128  # d2 constant for n=2
    x_bar = x.mean()
    ucl = x_bar + 3 * sigma
    lcl = max(0, x_bar - 3 * sigma)
    return x_bar, sigma, ucl, lcl

network = df[df["category"] == "Network"]["resolution_hours"].values
x_bar, sigma, ucl, lcl = imr_limits(network)

print(f"Center line : {x_bar:.1f}h")
print(f"UCL         : {ucl:.1f}h")
print(f"LCL         : {lcl:.1f}h")
print(f"Out-of-control points: {(network > ucl).sum()} of {len(network)}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(network, marker="o", markersize=2, linewidth=0.8, color="#4C72B0")
ax.axhline(x_bar, color="black", linewidth=1, label="Center line")
ax.axhline(ucl, color="#C44E52", linestyle="--", linewidth=1, label="UCL")
ax.axhline(lcl, color="#C44E52", linestyle="--", linewidth=1)
ax.axhline(SLA_HOURS, color="#55A868", linestyle=":", linewidth=1.5, label="SLA target")
ax.set_title("Network -- Resolution Time Control Chart (Baseline)", fontsize=12, pad=10)
ax.set_xlabel("Ticket sequence")
ax.set_ylabel("Resolution time (hours)")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig(CHARTS / "control_chart_before.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sla_compliance = (network <= SLA_HOURS).mean() * 100
print(f"Baseline SLA compliance (Network): {sla_compliance:.1f}%")
print(f"Baseline mean MTTR (Network)     : {network.mean():.1f}h")